In [4]:
run_id ="c5e0cfca-1a07-4dd4-886a-a83a17bc1fb6"



StatementMeta(, 7c856011-d5ef-473a-b03a-15a0962c874a, 6, Finished, Available, Finished, False)

In [5]:
# ============================================================
# Loadind data into the monitoring.pipeline_run_summary
# This table aggregate pilepne run by activities 
# ============================================================

from pyspark.sql.functions import col

MONITORING_DB    = "monitoring"
LOG_TABLE        = f"{MONITORING_DB}.pipeline_run_log"
SUMMARY_TABLE    = f"{MONITORING_DB}.pipeline_run_summary"

summary_df = spark.sql(f"""


      SELECT  run_id,pipeline_name, workspace_name, pipeline_start ,pipeline_end,
              DATEDIFF(second, pipeline_start, pipeline_end) AS  total_duration_sec,
              total_activities ,succeeded,failed,skipped ,total_rows_read ,
              total_rows_written,overall_status,environment,triggered_by,log_date

      FROM (
        SELECT
            run_id,
            pipeline_name,
            workspace_name,
            MIN(start_time)                                        AS pipeline_start,
            MAX(end_time)                                          AS pipeline_end,
            COUNT(log_type)                                       AS total_activities,
            SUM(CASE WHEN status = 'Succeeded'  THEN 1 ELSE 0 END) AS succeeded,
            SUM(CASE WHEN status = 'Failed'     THEN 1 ELSE 0 END) AS failed,
            SUM(CASE WHEN status = 'Skipped'    THEN 1 ELSE 0 END) AS skipped,
            SUM(rows_read)                                         AS total_rows_read,
            SUM(rows_written)                                      AS total_rows_written,
            CASE WHEN SUM(CASE WHEN status = 'Failed' THEN 1 ELSE 0 END) > 0
                 THEN 'Failed' ELSE 'Succeeded'
            END                                                    AS overall_status,
            MAX(environment)                                       AS environment,
            MAX(triggered_by)                                      AS triggered_by,
            CAST(MAX(end_time) AS DATE)                            AS log_date
        FROM {LOG_TABLE}
        WHERE run_id = '{run_id}'
        GROUP BY run_id, pipeline_name, workspace_name
            )t
    """)


summary_df = summary_df.withColumn("total_activities", col("total_activities").cast("int")) \
                       .withColumn("succeeded", col("succeeded").cast("int")) \
                       .withColumn("failed", col("failed").cast("int")) \
                       .withColumn("skipped", col("skipped").cast("int")) \
       


display (summary_df)

# =========================
# C Write to existing table
# =========================
summary_df.write \
        .format("delta") \
       .mode("append") \
       .option("replaceWhere", f"run_id = '{run_id}'") \
       .saveAsTable(SUMMARY_TABLE)



StatementMeta(, 7c856011-d5ef-473a-b03a-15a0962c874a, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c0dadd64-5402-4c1f-9232-b0c12cbfd0c2)

✅ Summary updated for run_id: c5e0cfca-1a07-4dd4-886a-a83a17bc1fb6
